In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# makin assumptions, not real data
COST_MISSED_FRAUD = 100
COST_FALSE_ALARM = 2
os.chdir("C:\\imp stuff\\credit car fraou\\data")


In [ ]:
df = pd.read_csv("creditcard.csv")

# if Class loaded as text like 0, convert it to a number
if not pd.api.types.is_numeric_dtype(df["Class"]):
    df["Class"] = df["Class"].astype(str).str.strip("'\"").astype(int)

# Drop exact duplicate rows
df = df.drop_duplicates()

print("Rows after dropping duplicates:", len(df))
print("Frauds:", int(df["Class"].sum()))

Rows after dropping duplicates: 283726
Frauds: 473


In [4]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Test rows: {len(X_test):,}   Test frauds: {int(y_test.sum())}")

Test rows: 56,746   Test frauds: 95


In [5]:
assert os.path.exists("test_predictions.csv"), (
    "test_predictions.csv not found. Re-run the last cells of notebook 05 first."
)
preds = pd.read_csv("test_predictions.csv")

# Check 1: same number of rows
assert len(preds) == len(X_test), f"Row count differs: {len(preds)} vs {len(X_test)}"

# Check 2: the true answers match in the same order
assert (preds["Class"].values == y_test.values).all(), (
    "Class column does not match. The split settings may differ from notebook 02."
)

print("Check passed: the predictions line up with the rebuilt test rows.")

Check passed: the predictions line up with the rebuilt test rows.


In [ ]:
AMOUNT_BINS = [-0.01, 1, 10, 50, 100, 250, 500, 1000, np.inf]
AMOUNT_LABELS = ["<=1", "1-10", "10-50", "50-100", "100-250", "250-500", "500-1000", "1000+"]


def add_features(frame):
    # Build a new table with the same row numbers as the input
    out = pd.DataFrame(index=frame.index)
    out["source_row"] = frame.index                              # original row number in creditcard.csv
    out["Time"] = frame["Time"]                                  # seconds since the first transaction
    out["HourOfDay"] = ((frame["Time"] // 3600) % 24).astype(int)   # // = divide and round down, % 24 wraps to a 24h clock
    out["Day"] = (frame["Time"] // 86400).astype(int) + 1        # 86400 seconds in a day; +1 so days are 1 and 2
    out["Amount"] = frame["Amount"]

    # sorts each Amount into one of the buckets
    bucket = pd.cut(frame["Amount"], bins=AMOUNT_BINS, labels=AMOUNT_LABELS)
    out["AmountBucket"] = bucket.astype(str)
    out["AmountBucketOrder"] = bucket.cat.codes + 1              # codes are 0-7; +1 makes them 1-8
    return out

In [7]:
all_tx = add_features(df)
all_tx["Class"] = df["Class"]
all_tx["ActualLabel"] = np.where(all_tx["Class"] == 1, "Fraud", "Legit")

print(all_tx.shape)
all_tx.head()

(283726, 9)


,source_row,Time,HourOfDay,Day,Amount,AmountBucket,AmountBucketOrder,Class,ActualLabel
0,0,0.0,0,1,149.62,100-250,5,0,Legit
1,1,0.0,0,1,2.69,1-10,2,0,Legit
2,2,1.0,0,1,378.66,250-500,6,0,Legit
3,3,1.0,0,1,123.50,100-250,5,0,Legit
4,4,2.0,0,1,69.99,50-100,4,0,Legit


In [8]:
test_tbl = add_features(X_test)
test_tbl["Class"] = y_test.values
test_tbl["ActualLabel"] = np.where(test_tbl["Class"] == 1, "Fraud", "Legit")

# .values attaches the predictions by position (the safety check above proved the order matches)
test_tbl["fraud_probability"] = preds["fraud_probability"].values.round(6)
test_tbl["flagged"] = preds["flagged"].values

is_fraud = test_tbl["Class"] == 1
is_flagged = test_tbl["flagged"] == 1

# np.select checks each condition in turn and gives the matching label
test_tbl["Outcome"] = np.select(
    [is_fraud & is_flagged, ~is_fraud & is_flagged, is_fraud & ~is_flagged],
    ["Caught fraud", "False alarm", "Missed fraud"],
    default="Correctly passed",
)

print(test_tbl["Outcome"].value_counts())

Outcome
Correctly passed    56584
Caught fraud           79
False alarm            67
Missed fraud           16
Name: count, dtype: int64


In [9]:
prob = test_tbl["fraud_probability"].values
truth = test_tbl["Class"].values
n_total = len(truth)

rows = []
# np.arange(0.01, 1.00, 0.01) gives 0.01, 0.02, ... 0.99. round(2) removes tiny decimal errors.
for t in np.round(np.arange(0.01, 1.00, 0.01), 2):
    pred = prob >= t                              # True where we would flag fraud at this threshold
    tp = int((pred & (truth == 1)).sum())         # fraud caught
    fp = int((pred & (truth == 0)).sum())         # false alarms
    fn = int((~pred & (truth == 1)).sum())        # fraud missed
    rows.append({
        "threshold": t,
        "frauds_caught": tp,
        "frauds_missed": fn,
        "false_alarms": fp,
        "flagged": tp + fp,
        "flagged_pct": round((tp + fp) / n_total * 100, 4),
        "precision": round(tp / (tp + fp), 4) if (tp + fp) > 0 else np.nan,
        "recall": round(tp / (tp + fn), 4),
        "total_cost": fn * COST_MISSED_FRAUD + fp * COST_FALSE_ALARM,
        "cost_no_model": int(truth.sum()) * COST_MISSED_FRAUD,
    })

threshold_tbl = pd.DataFrame(rows)

# Show a few rows, including the one for your chosen threshold (0.33)
threshold_tbl[threshold_tbl["threshold"].isin([0.1, 0.33, 0.5, 0.9])]

,threshold,frauds_caught,frauds_missed,false_alarms,flagged,flagged_pct,precision,recall,total_cost,cost_no_model
9,0.10,80,15,211,291,0.5128,0.2749,0.8421,1922,9500
32,0.33,79,16,67,146,0.2573,0.5411,0.8316,1734,9500
49,0.50,77,18,30,107,0.1886,0.7196,0.8105,1860,9500
89,0.90,71,24,4,75,0.1322,0.9467,0.7474,2408,9500


In [11]:
all_tx.to_csv("powerbi_all_transactions.csv", index=False)
test_tbl.to_csv("powerbi_test_results.csv", index=False)
threshold_tbl.to_csv("powerbi_threshold_table.csv", index=False)

print("Saved to the powerbi_data folder:")
print(f"  powerbi_all_transactions.csv  {len(all_tx):,} rows")
print(f"  powerbi_test_results.csv      {len(test_tbl):,} rows")
print(f"  powerbi_threshold_table.csv   {len(threshold_tbl):,} rows")

Saved to the powerbi_data folder:
  powerbi_all_transactions.csv  283,726 rows
  powerbi_test_results.csv      56,746 rows
  powerbi_threshold_table.csv   99 rows
